# BERT-Based Binary Sentiment Analysis — Experiments Notebook

This notebook drives the reusable functions in `src/` to run the full experiment pipeline: dataset understanding, three model implementations, hyperparameter search, model comparison, final evaluation, and error analysis.

**No results in this notebook are pre-filled.** Every cell must be executed locally (after placing `data/sentiment_train.csv`) to produce real output. Expensive cells (BERT training, hyperparameter search) are clearly marked.

## 1. Project Overview

This project compares three approaches to binary sentiment classification:

1. TF-IDF + Logistic Regression (classical baseline)
2. Frozen BERT + classification head (pretrained representation)
3. Fully fine-tuned BERT (task-specific fine-tuning)

All reusable logic lives in `src/`; this notebook only orchestrates it and visualizes results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.seed import set_seed
from src.data import load_raw_dataset, validate_dataset, summarize_dataset, split_dataset
from src.preprocessing import load_tokenizer, compute_token_length_stats, select_max_length

set_seed(config.SEED)
print('Device:', config.DEVICE)

## 2. Problem Statement

Given an English sentence, predict whether its sentiment is **Positive** or **Negative**. This is a binary classification problem evaluated with accuracy, precision, recall, F1, and macro F1.

## 3. Dataset Overview

Load and validate `data/sentiment_train.csv`. All statistics below are computed live from your file — nothing is hardcoded.

In [ ]:
df = load_raw_dataset(config.DATA_PATH)
validate_dataset(df)
summary = summarize_dataset(df)
df.head()

## 4. Class Distribution

In [ ]:
label_counts = df[config.LABEL_COLUMN].value_counts().sort_index()
plt.figure(figsize=(5,4))
label_counts.plot(kind='bar')
plt.title('Label Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
label_counts

## 5. Text-Length Analysis

Word-level length (for a quick sanity check) and BERT token-level length (used to pick `max_length` — see Section 9).

In [ ]:
word_lens = df[config.TEXT_COLUMN].dropna().astype(str).apply(lambda s: len(s.split()))
plt.figure(figsize=(6,4))
plt.hist(word_lens, bins=30)
plt.title('Text Length (words)')
plt.xlabel('Words per sentence')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
word_lens.describe()

## 6. Train/Validation/Test Split

Stratified 70/15/15 split with a fixed seed (42). The test set below is never touched again until Section 14 (Final Test Evaluation).

In [ ]:
splits = split_dataset(df)
print('Train:', len(splits.train_text))
print('Val  :', len(splits.val_text))
print('Test :', len(splits.test_text))

## 7. TF-IDF + Logistic Regression

Model 1: classical baseline. This cell is cheap to run.

In [ ]:
from src.baseline import train_and_evaluate as train_tfidf

tfidf_pipeline, tfidf_result = train_tfidf()
tfidf_result['metrics']

## 8. Frozen BERT

Model 2: only the classification head is trained; BERT parameters are frozen. **This cell downloads `bert-base-uncased` and trains a small head — cheap relative to full fine-tuning, but still requires the dataset and a few minutes of compute.**

In [ ]:
from src.train import train_one_config
from src.bert_model import save_model
from src.preprocessing import load_tokenizer, compute_token_length_stats, select_max_length

tokenizer = load_tokenizer()
token_stats = compute_token_length_stats(splits.train_text, tokenizer)
max_length = select_max_length(token_stats)

frozen_cfg = config.TrainingConfig(freeze_bert=True, epochs=4, learning_rate=2e-5)
frozen_result = train_one_config(frozen_cfg, splits, tokenizer, max_length)
save_model(frozen_result['model'], tokenizer, config.FROZEN_BERT_DIR, max_length,
           extra_config={'best_val_f1': frozen_result['best_val_f1']})

## 9. Fully Fine-Tuned BERT

Model 3: the main model. All BERT parameters are trainable. **This is the most expensive cell in the notebook — run it locally, not on the free Claude tier.**

In [ ]:
finetune_cfg = config.TrainingConfig(freeze_bert=False, epochs=4, learning_rate=2e-5)
finetune_result = train_one_config(finetune_cfg, splits, tokenizer, max_length)
save_model(finetune_result['model'], tokenizer, config.BEST_BERT_DIR, max_length,
           extra_config={'best_val_f1': finetune_result['best_val_f1']})

## 10. Learning-Rate Experiments

Searches `[1e-5, 2e-5, 3e-5, 5e-5]`, selecting the best by validation F1. **Expensive — trains 4 full models.** Writes `experiments/learning_rate_results.csv`.

In [ ]:
from src.train import run_lr_search

lr_results = run_lr_search(splits, tokenizer, max_length, finetune_cfg)
lr_results

## 11. Epoch Experiments

Searches `[2, 3, 4, 5]` epochs using the best learning rate from Section 10. **Expensive — trains up to 4 more models.** Writes `experiments/epoch_results.csv`.

In [ ]:
from src.train import run_epoch_search
from dataclasses import replace

best_lr = lr_results.loc[lr_results['best_validation_f1'].idxmax(), 'learning_rate']
epoch_search_cfg = replace(finetune_cfg, learning_rate=best_lr)
epoch_results = run_epoch_search(splits, tokenizer, max_length, epoch_search_cfg)
epoch_results

## 12. Training Curves

In [ ]:
from src.evaluate import plot_training_curves

plot_training_curves(finetune_result['history'])
from IPython.display import Image
Image(config.TRAINING_CURVES_PATH)

## 13. Model Comparison

Compares all three saved models on the untouched test set.

In [ ]:
from src.evaluate import compare_all_models

comparison_df = compare_all_models()
comparison_df

## 14. Final Test Evaluation

Evaluates the selected final model (fully fine-tuned BERT) on the test set. This is the only cell that should touch the test set for the final model.

In [ ]:
from src.evaluate import final_test_evaluation

final_result = final_test_evaluation(config.BEST_BERT_DIR)
final_result['metrics']

## 15. Confusion Matrix

In [ ]:
from IPython.display import Image
Image(config.CONFUSION_MATRIX_PATH)

## 16. Error Analysis

In [ ]:
from src.error_analysis import run_error_analysis

error_df = run_error_analysis(config.BEST_BERT_DIR)
error_df[~error_df['correct']].head(10)

## 17. Conclusions

`To be written after running the full pipeline above.` Summarize:

- Which model achieved the best test-set F1 / macro F1
- Whether fine-tuning BERT improved meaningfully over the frozen baseline
- The dominant error patterns found in Section 16
- Any practical trade-offs (training time vs. performance) worth noting for the README results table